# Mô hình TF-IDF — Tìm kiếm cấp độ Bài viết (Document-level) - Đa Cấu Hình



## Bước 1: Khởi tạo, nạp Thư viện và Định vị Thư mục Dự án

In [1]:
import sys
import os
import time
import re
import unicodedata
import string
import pickle
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import save_npz, load_npz

# Thiết lập console encoding UTF-8 khi chạy trên Windows để tránh lỗi ký tự Unicode
if sys.platform.startswith('win') and hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Tự động định vị thư mục gốc dự án chứa thư mục 'data'
project_root = os.path.abspath(os.getcwd())
while project_root != os.path.dirname(project_root):
    if os.path.exists(os.path.join(project_root, 'data', 'news_dataset_df_prep2.pkl')):
        break
    if os.path.exists(os.path.join(project_root, 'cuoi_ky', 'data', 'news_dataset_df_prep2.pkl')):
        project_root = os.path.join(project_root, 'cuoi_ky')
        break
    project_root = os.path.dirname(project_root)
else:
    project_root = os.path.abspath(os.getcwd())

print(f"OK: Thư mục dự án được xác định là: {project_root}")

OK: Thư mục dự án được xác định là: d:\UTE\NAM3\k2_dot_2\xu_ly_ngon_ngu_tu_nhien\cuoi_ky


## Bước 2: Nạp dữ liệu DataFrame bài báo làm Metadata

In [2]:
DATA_PATH = os.path.join(project_root, 'data', 'news_dataset_df_prep2.pkl')

print(f"TIÊN TRÌNH: Đang tải dữ liệu bài báo làm metadata từ: {DATA_PATH}...")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"LỖI: Không tìm thấy file dữ liệu tại {DATA_PATH}")

df_load = pd.read_pickle(DATA_PATH)
print(f"OK: Tải thành công {len(df_load)} bài báo làm metadata tìm kiếm.")

TIÊN TRÌNH: Đang tải dữ liệu bài báo làm metadata từ: d:\UTE\NAM3\k2_dot_2\xu_ly_ngon_ngu_tu_nhien\cuoi_ky\data\news_dataset_df_prep2.pkl...
OK: Tải thành công 160254 bài báo làm metadata tìm kiếm.


## Bước 3: Định nghĩa các hàm Tiền xử lý văn bản (Tách từ & Làm sạch)

In [3]:
_rdrsegmenter = None

def get_rdrsegmenter():
    global _rdrsegmenter
    if _rdrsegmenter is None:
        import py_vncorenlp
        data_dir = os.path.join(project_root, "data")
        jar_path = os.path.join(data_dir, "VnCoreNLP-1.2.jar")
        if not os.path.exists(jar_path):
            raise FileNotFoundError(f"LỖI: VnCoreNLP jar không tồn tại tại: {jar_path}")
        _rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=data_dir)
    return _rdrsegmenter

def normalize_date(text):
    pattern_dmy = r"\b(\d{1,2})[-/.](\d{1,2})[-/.](\d{2,4})\b"
    pattern_dm = r"\b(\d{1,2})[-/.](\d{1,2})\b"
    def repl_dmy(m):
        day, month, year = m.groups()
        if len(year) == 2:
            year = "20" + year
        return f" DATE{year}{int(month):02d}{int(day):02d} "
    def repl_dm(m):
        day, month = m.groups()
        return f" DATE{int(month):02d}{int(day):02d} "
    text = re.sub(pattern_dmy, repl_dmy, text)
    text = re.sub(pattern_dm, repl_dm, text)
    return text

def clean_text_keep_case(text):
    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', ' ', text)
    text = normalize_date(text)
    custom_punctuation = string.punctuation.replace('_', '').replace('-', '')
    translator = str.maketrans(custom_punctuation, ' ' * len(custom_punctuation))
    text = text.translate(translator)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_vn_accents(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFD', text)
    text = ''.join([c for c in text if unicodedata.category(c) != 'Mn'])
    text = text.replace('đ', 'd').replace('Đ', 'D')
    return text

def load_stopwords():
    stopwords_path = os.path.join(project_root, "data", "vietnamese-stopwords.txt")
    if not os.path.exists(stopwords_path):
        return set()
    with open(stopwords_path, "r", encoding="utf-8") as f:
        stopwords = set([line.strip().replace(' ', '_') for line in f])
    return stopwords

def remove_stopwords_func(text, stopwords_set):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""
    tokens = text.split()
    filtered_tokens = [t for t in tokens if t.lower() not in stopwords_set]
    return " ".join(filtered_tokens)

def preprocess_pipeline(text, use_accent=True, remove_stopwords=False, stopwords_set=None):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""
    cleaned = clean_text_keep_case(text)
    if not cleaned:
        return ""
    segmenter = get_rdrsegmenter()
    segmented_sentences = segmenter.word_segment(cleaned)
    segmented_text = " ".join(segmented_sentences)
    segmented_text = segmented_text.lower()
    if remove_stopwords:
        if stopwords_set is None:
            stopwords_set = load_stopwords()
        segmented_text = remove_stopwords_func(segmented_text, stopwords_set)
    if not use_accent:
        segmented_text = remove_vn_accents(segmented_text)
    return segmented_text

# Tải danh sách từ dừng vào bộ nhớ
stopwords_set = load_stopwords()
print(f"OK: Đã nạp danh sách stopwords ({len(stopwords_set)} từ) thành công!")

OK: Đã nạp danh sách stopwords (1942 từ) thành công!


## Bước 4: Thiết lập cấu hình Đường dẫn & Hàm bổ trợ chỉ mục

In [4]:
MODELS_DIR = os.path.join(project_root, 'models')
INDEXES_DIR = os.path.join(project_root, 'indexes', 'tfidf')

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(INDEXES_DIR, exist_ok=True)

def get_config_name(use_accent, remove_stopwords):
    accent_str = "accented" if use_accent else "unaccented"
    stopwords_str = "no_stopwords" if remove_stopwords else "with_stopwords"
    return f"{accent_str}_{stopwords_str}"

def get_model_paths(config_name):
    vec_path = os.path.join(MODELS_DIR, f"tfidf_vectorizer_{config_name}.pkl")
    mat_path = os.path.join(MODELS_DIR, f"tfidf_matrix_{config_name}.npz")
    idx_path = os.path.join(INDEXES_DIR, f"inverted_index_{config_name}.pkl")
    return vec_path, mat_path, idx_path

def build_inverted_index(corpus, doc_ids):
    inverted_index = defaultdict(list)
    for idx, doc_id in enumerate(doc_ids):
        text = corpus[idx]
        if pd.isna(text):
            continue
        tokens = str(text).split()
        unique_tokens = set(tokens)
        for token in unique_tokens:
            inverted_index[token].append(doc_id)
    return dict(inverted_index)

print("OK: Đã sẵn sàng các cấu hình đường dẫn và hàm tạo Inverted Index!")

OK: Đã sẵn sàng các cấu hình đường dẫn và hàm tạo Inverted Index!


## Bước 5: Định nghĩa hàm Huấn luyện TF-IDF

In [5]:
def train_tfidf_config(use_accent, remove_stopwords, stopwords_set):
    config_name = get_config_name(use_accent, remove_stopwords)
    vec_path, mat_path, idx_path = get_model_paths(config_name)
    
    print(f"TIÊN TRÌNH: Bắt đầu huấn luyện TF-IDF cho cấu hình: {config_name}")
    
    # Nạp dữ liệu từ file tương ứng đã được xử lý sẵn để tối ưu hóa hiệu năng
    if remove_stopwords:
        data_file = os.path.join(project_root, 'data', 'news_dataset_df_prep3.pkl')
    else:
        data_file = os.path.join(project_root, 'data', 'news_dataset_df_prep2.pkl')
        
    print(f"TIÊN TRÌNH: Đang nạp dữ liệu huấn luyện từ {data_file}...")
    df_train = pd.read_pickle(data_file)
    
    # Chọn cột tương ứng đã tách từ
    if use_accent:
        corpus = df_train['combined_processed'].fillna("").astype(str).tolist()
    else:
        if 'combined_unaccented' in df_train.columns:
            corpus = df_train['combined_unaccented'].fillna("").astype(str).tolist()
        else:
            print("TIÊN TRÌNH: Đang tự động loại bỏ dấu khỏi cột combined_processed...")
            corpus = df_train['combined_processed'].fillna("").astype(str).tolist()
            corpus = [remove_vn_accents(text) for text in corpus]
            
    # Khởi tạo TfidfVectorizer (tương tự tf_idf.ipynb)
    vectorizer = TfidfVectorizer(
        min_df=5,
        max_df=0.8,
        max_features=100000,
        lowercase=True
    )
    
    print("TIÊN TRÌNH: Đang tính toán ma trận TF-IDF (fit_transform)... (Quá trình này có thể mất 1-2 phút)")
    tfidf_matrix = vectorizer.fit_transform(corpus)
    
    print(f"OK: Kích thước từ vựng = {len(vectorizer.vocabulary_)} | Shape ma trận = {tfidf_matrix.shape}")
    
    # Lưu mô hình và ma trận thưa
    with open(vec_path, 'wb') as f:
        pickle.dump(vectorizer, f)
    save_npz(mat_path, tfidf_matrix)
    print(f"OK: Đã lưu Vectorizer và Ma trận TF-IDF vào thư mục '{MODELS_DIR}'")
    
    # Xây dựng và lưu Inverted Index để tìm kiếm nhanh
    print("TIÊN TRÌNH: Đang xây dựng chỉ mục ngược Inverted Index...")
    inverted_index = build_inverted_index(corpus, df_train['id'].tolist())
    with open(idx_path, 'wb') as f:
        pickle.dump(inverted_index, f)
    print(f"OK: Đã lưu Inverted Index vào thư mục '{INDEXES_DIR}'")
    
    print(f"OK: Hoàn tất huấn luyện cấu hình {config_name}!\n")

## Bước 6: Huấn luyện / Kiểm tra các cấu hình cụ thể

Chúng ta chạy độc lập từng cell cho mỗi cấu hình để dễ kiểm soát tiến trình. 
Đối với cấu hình **Có dấu + Có stopword (Mặc định)**, hệ thống sẽ ưu tiên tải trực tiếp từ các file pre-trained gốc có sẵn.

In [6]:
# --- Cấu hình 1: Có dấu + Có stopword (Mặc định) ---
use_accent = True
remove_stopwords = False
config_name = get_config_name(use_accent, remove_stopwords)
vec_path, mat_path, idx_path = get_model_paths(config_name)

fallback_vec = os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl")
fallback_mat = os.path.join(MODELS_DIR, "tfidf_matrix.npz")
fallback_idx = os.path.join(project_root, "data", "inverted_index.pkl")

if os.path.exists(fallback_vec) and os.path.exists(fallback_mat) and os.path.exists(fallback_idx):
    print("OK: Cấu hình mặc định (Có dấu + Có stopword) đã có sẵn các file pre-trained gốc. Bỏ qua bước huấn luyện.")
elif os.path.exists(vec_path) and os.path.exists(mat_path) and os.path.exists(idx_path):
    print(f"OK: Cấu hình {config_name} đã được huấn luyện và lưu trữ trước đó.")
else:
    train_tfidf_config(use_accent, remove_stopwords, stopwords_set)

OK: Cấu hình mặc định (Có dấu + Có stopword) đã có sẵn các file pre-trained gốc. Bỏ qua bước huấn luyện.


In [7]:
# --- Cấu hình 2: Có dấu + Không stopword ---
use_accent = True
remove_stopwords = True
config_name = get_config_name(use_accent, remove_stopwords)
vec_path, mat_path, idx_path = get_model_paths(config_name)

if os.path.exists(vec_path) and os.path.exists(mat_path) and os.path.exists(idx_path):
    print(f"OK: Cấu hình {config_name} đã được huấn luyện và lưu trữ trước đó.")
else:
    train_tfidf_config(use_accent, remove_stopwords, stopwords_set)

OK: Cấu hình accented_no_stopwords đã được huấn luyện và lưu trữ trước đó.


In [8]:
# --- Cấu hình 3: Không dấu + Có stopword ---
use_accent = False
remove_stopwords = False
config_name = get_config_name(use_accent, remove_stopwords)
vec_path, mat_path, idx_path = get_model_paths(config_name)

if os.path.exists(vec_path) and os.path.exists(mat_path) and os.path.exists(idx_path):
    print(f"OK: Cấu hình {config_name} đã được huấn luyện và lưu trữ trước đó.")
else:
    train_tfidf_config(use_accent, remove_stopwords, stopwords_set)

OK: Cấu hình unaccented_with_stopwords đã được huấn luyện và lưu trữ trước đó.


In [9]:
# --- Cấu hình 4: Không dấu + Không stopword ---
use_accent = False
remove_stopwords = True
config_name = get_config_name(use_accent, remove_stopwords)
vec_path, mat_path, idx_path = get_model_paths(config_name)

if os.path.exists(vec_path) and os.path.exists(mat_path) and os.path.exists(idx_path):
    print(f"OK: Cấu hình {config_name} đã được huấn luyện và lưu trữ trước đó.")
else:
    train_tfidf_config(use_accent, remove_stopwords, stopwords_set)

OK: Cấu hình unaccented_no_stopwords đã được huấn luyện và lưu trữ trước đó.


## Bước 7: Định nghĩa các hàm Tải mô hình & Tìm kiếm

In [10]:
_loaded_models = {}

def get_tfidf_model(use_accent, remove_stopwords):
    config_name = get_config_name(use_accent, remove_stopwords)
    if config_name in _loaded_models:
        return _loaded_models[config_name]
        
    vec_path, mat_path, idx_path = get_model_paths(config_name)
    
    # Fallback nạp các file gốc của cấu hình mặc định (Có dấu + Có stopword)
    if use_accent and not remove_stopwords:
        fallback_vec = os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl")
        fallback_mat = os.path.join(MODELS_DIR, "tfidf_matrix.npz")
        fallback_idx = os.path.join(project_root, "data", "inverted_index.pkl")
        if os.path.exists(fallback_vec) and os.path.exists(fallback_mat) and os.path.exists(fallback_idx):
            vec_path, mat_path, idx_path = fallback_vec, fallback_mat, fallback_idx
            print("OK: Đang sử dụng các file pre-trained gốc cho cấu hình mặc định.")
            
    # Huấn luyện tự động nếu thiếu file
    if not (os.path.exists(vec_path) and os.path.exists(mat_path) and os.path.exists(idx_path)):
        print(f"Luu y: Phát hiện thiếu mô hình cho cấu hình {config_name}. Đang huấn luyện tự động...")
        train_tfidf_config(use_accent, remove_stopwords, stopwords_set)
        if use_accent and not remove_stopwords and not os.path.exists(vec_path):
            vec_path, mat_path, idx_path = get_model_paths(config_name)
            
    with open(vec_path, 'rb') as f:
        vectorizer = pickle.load(f)
    tfidf_matrix = load_npz(mat_path)
    with open(idx_path, 'rb') as f:
        inverted_index = pickle.load(f)
        
    # Tạo bộ ánh xạ vị trí từ doc_id sang index dòng trong ma trận để tăng tốc tìm kiếm
    doc_id_to_idx = {doc_id: idx for idx, doc_id in enumerate(df_load['id'])}
    
    _loaded_models[config_name] = (vectorizer, tfidf_matrix, inverted_index, doc_id_to_idx)
    return _loaded_models[config_name]

def search_tfidf(query, k=5, use_accent=True, remove_stopwords=False):
    vectorizer, tfidf_matrix, inverted_index, doc_id_to_idx = get_tfidf_model(use_accent, remove_stopwords)
    
    # Tiền xử lý câu truy vấn thô của người dùng
    p_stopwords_set = stopwords_set if remove_stopwords else None
    processed_query = preprocess_pipeline(
        query,
        use_accent=True,
        remove_stopwords=remove_stopwords,
        stopwords_set=p_stopwords_set
    )
    if not use_accent:
        processed_query = remove_vn_accents(processed_query)
        
    # GIAI ĐOẠN 1: Lọc thô bằng Inverted Index (giống tf_idf.ipynb)
    query_tokens = vectorizer.build_analyzer()(processed_query)
    candidate_doc_ids = set()
    for token in query_tokens:
        if token in inverted_index:
            candidate_doc_ids.update(inverted_index[token])
            
    if not candidate_doc_ids:
        return []
        
    candidate_indices = [doc_id_to_idx[d_id] for d_id in candidate_doc_ids if d_id in doc_id_to_idx]
    
    # GIAI ĐOẠN 2: Xếp hạng chính xác bằng Cosine Similarity
    query_vector = vectorizer.transform([processed_query])
    sub_tfidf_matrix = tfidf_matrix[candidate_indices]
    similarities = cosine_similarity(query_vector, sub_tfidf_matrix).flatten()
    
    top_k_sub_indices = similarities.argsort()[-k:][::-1]
    results = []
    for rank, sub_idx in enumerate(top_k_sub_indices):
        score = similarities[sub_idx]
        if score <= 0:
            continue
        original_idx = candidate_indices[sub_idx]
        row = df_load.iloc[original_idx]
        results.append({
            'rank': rank + 1,
            'score': float(score),
            'doc_id': row['id'],
            'title': row['title'],
            'source': row.get('source', ''),
            'topic': row.get('topic', ''),
            'url': row.get('url', ''),
            'text': row.get('content', '')
        })
    return results

print("OK: Các hàm tìm kiếm đã sẵn sàng hoạt động!")

OK: Các hàm tìm kiếm đã sẵn sàng hoạt động!


## Bước 8: Kiểm thử và Đánh giá hiệu năng giữa các cấu hình

In [11]:
query = "Vụ cướp tiệm vàng ở Huế bằng súng AK"
print(f"Câu truy vấn gốc: '{query}'\n")

configs = [
    ("Có dấu + Có stopword",       True,  False),
    ("Có dấu + Không stopword",    True,  True),
    ("Không dấu + Có stopword",    False, False),
    ("Không dấu + Không stopword", False, True),
]

rows = []
for label, use_acc, rem_stop in configs:
    processed = preprocess_pipeline(query, use_accent=use_acc, remove_stopwords=rem_stop)
    rows.append({"Cấu hình": label, "Kết quả tiền xử lý": processed})

display(pd.DataFrame(rows))

Câu truy vấn gốc: 'Vụ cướp tiệm vàng ở Huế bằng súng AK'



,Cấu hình,Kết quả tiền xử lý
0,Có dấu + Có stopword,vụ cướp tiệm vàng ở huế bằng súng ak
1,Có dấu + Không stopword,vụ cướp tiệm vàng huế súng ak
2,Không dấu + Có stopword,vu cuop tiem vang o hue bang sung ak
3,Không dấu + Không stopword,vu cuop tiem vang hue sung ak


In [12]:
def display_tfidf_results(results, title):
    """Hiển thị kết quả tìm kiếm TF-IDF dưới dạng bảng."""
    print(f"\n{'='*20} {title} {'='*20}")
    if not results:
        print("Luu y: Không tìm thấy kết quả nào.")
        return
    df_res = pd.DataFrame(results)
    if 'text' in df_res.columns:
        df_res['text_snippet'] = df_res['text'].str.slice(0, 150) + '...'
    cols = [c for c in ['rank', 'score', 'doc_id', 'title', 'topic', 'text_snippet'] if c in df_res.columns]
    display(df_res[cols])

In [13]:
TOP_K = 5
for label, use_acc, rem_stop in configs:
    t0 = time.time()
    results = search_tfidf(query, k=TOP_K, use_accent=use_acc, remove_stopwords=rem_stop)
    elapsed = time.time() - t0
    display_tfidf_results(results, f"TF-IDF | {label} | Thoi gian: {elapsed:.3f}s")

OK: Đang sử dụng các file pre-trained gốc cho cấu hình mặc định.

==================== TF-IDF | Có dấu + Có stopword | Thoi gian: 9.020s ====================


,rank,score,doc_id,title,topic,text_snippet
0,1,0.629572,215798,Kẻ mang AK cướp tiệm vàng ở Huế đòi gặp phó gi...,Pháp luật,"Trưa 31/7, Công an tỉnh Thừa Thiên - Huế phong..."
1,2,0.593850,216212,Thừa Thiên - Huế: Bắt nghi phạm dùng súng AK c...,Tin nhanh 24h,Video: Cận cảnh hiện trường nghi phạm vác súng...
2,3,0.579795,216062,Nghi phạm dùng súng AK cướp tiệm vàng ở TT- Hu...,Thời sự,Nghi phạm dùng súng AK cướp tiệm vàng ở TT- Hu...
3,4,0.575877,217036,Mang súng AK cướp tiệm vàng ở Huế: Lời kể hãi ...,Tin nhanh 24h,Hơn 5 tiếng đồng hồ kể từ thời điểm Ngô Văn Qu...
4,5,0.575727,216796,Người đàn ông cầm súng AK cướp tiệm vàng ở Huế,,Cập nhật: Một lãnh đạo công an tỉnh Thừa Thiên...



==================== TF-IDF | Có dấu + Không stopword | Thoi gian: 5.120s ====================


,rank,score,doc_id,title,topic,text_snippet
0,1,0.652899,215798,Kẻ mang AK cướp tiệm vàng ở Huế đòi gặp phó gi...,Pháp luật,"Trưa 31/7, Công an tỉnh Thừa Thiên - Huế phong..."
1,2,0.611689,216212,Thừa Thiên - Huế: Bắt nghi phạm dùng súng AK c...,Tin nhanh 24h,Video: Cận cảnh hiện trường nghi phạm vác súng...
2,3,0.610402,216796,Người đàn ông cầm súng AK cướp tiệm vàng ở Huế,,Cập nhật: Một lãnh đạo công an tỉnh Thừa Thiên...
3,4,0.607470,216062,Nghi phạm dùng súng AK cướp tiệm vàng ở TT- Hu...,Thời sự,Nghi phạm dùng súng AK cướp tiệm vàng ở TT- Hu...
4,5,0.605536,217036,Mang súng AK cướp tiệm vàng ở Huế: Lời kể hãi ...,Tin nhanh 24h,Hơn 5 tiếng đồng hồ kể từ thời điểm Ngô Văn Qu...



==================== TF-IDF | Không dấu + Có stopword | Thoi gian: 4.994s ====================


,rank,score,doc_id,title,topic,text_snippet
0,1,0.647843,215798,Kẻ mang AK cướp tiệm vàng ở Huế đòi gặp phó gi...,Pháp luật,"Trưa 31/7, Công an tỉnh Thừa Thiên - Huế phong..."
1,2,0.610889,216212,Thừa Thiên - Huế: Bắt nghi phạm dùng súng AK c...,Tin nhanh 24h,Video: Cận cảnh hiện trường nghi phạm vác súng...
2,3,0.592888,216062,Nghi phạm dùng súng AK cướp tiệm vàng ở TT- Hu...,Thời sự,Nghi phạm dùng súng AK cướp tiệm vàng ở TT- Hu...
3,4,0.589987,216795,Công an Huế vận động người dân trả lại vàng do...,Trong nước,"Chiều 31-7, Công an TP Huế có thông báo đề ngh..."
4,5,0.586218,217036,Mang súng AK cướp tiệm vàng ở Huế: Lời kể hãi ...,Tin nhanh 24h,Hơn 5 tiếng đồng hồ kể từ thời điểm Ngô Văn Qu...



==================== TF-IDF | Không dấu + Không stopword | Thoi gian: 5.458s ====================


,rank,score,doc_id,title,topic,text_snippet
0,1,0.660784,215798,Kẻ mang AK cướp tiệm vàng ở Huế đòi gặp phó gi...,Pháp luật,"Trưa 31/7, Công an tỉnh Thừa Thiên - Huế phong..."
1,2,0.619692,216212,Thừa Thiên - Huế: Bắt nghi phạm dùng súng AK c...,Tin nhanh 24h,Video: Cận cảnh hiện trường nghi phạm vác súng...
2,3,0.602698,216062,Nghi phạm dùng súng AK cướp tiệm vàng ở TT- Hu...,Thời sự,Nghi phạm dùng súng AK cướp tiệm vàng ở TT- Hu...
3,4,0.601139,216795,Công an Huế vận động người dân trả lại vàng do...,Trong nước,"Chiều 31-7, Công an TP Huế có thông báo đề ngh..."
4,5,0.597123,217036,Mang súng AK cướp tiệm vàng ở Huế: Lời kể hãi ...,Tin nhanh 24h,Hơn 5 tiếng đồng hồ kể từ thời điểm Ngô Văn Qu...


In [14]:
summary_rows = []
for label, use_acc, rem_stop in configs:
    results = search_tfidf(query, k=1, use_accent=use_acc, remove_stopwords=rem_stop)
    if results:
        r = results[0]
        summary_rows.append({
            'Cấu hình': label,
            'Top-1 Score': f"{r['score']:.4f}",
            'Doc ID': r['doc_id'],
            'Title': r['title'][:80]
        })
    else:
        summary_rows.append({'Cấu hình': label, 'Top-1 Score': 'N/A', 'Doc ID': 'N/A', 'Title': 'Không có kết quả'})

display(pd.DataFrame(summary_rows))

,Cấu hình,Top-1 Score,Doc ID,Title
0,Có dấu + Có stopword,0.6296,215798,Kẻ mang AK cướp tiệm vàng ở Huế đòi gặp phó gi...
1,Có dấu + Không stopword,0.6529,215798,Kẻ mang AK cướp tiệm vàng ở Huế đòi gặp phó gi...
2,Không dấu + Có stopword,0.6478,215798,Kẻ mang AK cướp tiệm vàng ở Huế đòi gặp phó gi...
3,Không dấu + Không stopword,0.6608,215798,Kẻ mang AK cướp tiệm vàng ở Huế đòi gặp phó gi...


In [15]:
# 🔧 Thay đổi câu hỏi và các tham số dưới đây để chạy thử nghiệm tự do:
my_query = "giá xăng dầu trong nước hôm nay"
my_use_accent = True
my_remove_stopwords = False
my_top_k = 5

results = search_tfidf(my_query, k=my_top_k, use_accent=my_use_accent, remove_stopwords=my_remove_stopwords)
display_tfidf_results(results, f"Tìm kiếm Tương tác | Có dấu={my_use_accent}, Bỏ SW={my_remove_stopwords}")


==================== Tìm kiếm Tương tác | Có dấu=True, Bỏ SW=False ====================


,rank,score,doc_id,title,topic,text_snippet
0,1,0.606523,3628,Giá xăng dầu Malaysia chỉ 13.000 đồng/lít: Đại...,Kinh doanh,"Có thể nhập xăng dầu giá rẻ? Theo đó, ngày 3.6..."
1,2,0.592645,119488,"Giá xăng giảm hơn 3.000 đồng, vì sao giá bát p...",Kinh doanh,Giá xăng thời gian qua tăng liên tục lập tức k...
2,3,0.591040,122433,"Giá xăng giảm hơn 3.000 đồng, vì sao giá bát p...",Kinh tế,Giá xăng thời gian qua tăng liên tục lập tức k...
3,4,0.574408,111469,Giá xăng sắp giảm mạnh xuống dưới 30.000 đồng/...,Kinh tế,"Hiện nay, cơ cấu giá bán xăng dầu trong nước p..."
4,5,0.570784,97086,"Khẩn trương nghiên cứu thuế giá trị gia tăng, ...",Thị trường - Tiêu dùng,"Sáng 6/7, dưới sự chủ trì của Chủ tịch Quốc hộ..."
